# Thumbs_Robot: Unitree G1 Three-Digit Hand-Gesture Control via PPO
## Project Handbook & Grading Guide

This Notebook serves as the guided hands-on handbook for the CSCN8020 Reinforcement Learning Final Project. You can step-by-step execute the code directly in this notebook, completing the full workflow from environment setup, model audit, environment testing, model training, to results visualization.

### Project Overview & Team
- **Project Objective**: Use a continuous action Actor-Critic (PPO style) controller to control the 3-digit hand structure of the Unitree G1 robot (including two main fingers and one thumb). The goal is to achieve stable, smooth, and coordinated control in a MuJoCo simulation environment for three target gestures: Thumbs Up, Open/Stop, and Thumbs Down.
- **Academic Highlight**: Implement the **"Math MDP $\rightarrow$ Algorithm Logic $\rightarrow$ Code Variables $\rightarrow$ Real-time Logs" four-in-one precise alignment mapping** required by the professor.
- **Team Members**: Emmanuel • Liggia • Cemil • Chao

### 1. Environment Setup
To ensure that this project is fully reproducible on other machines, please execute the following cells in order. This will upgrade pip, install dependency packages, clone the official third-party Unitree repository, and verify the Python environment.

#### Step 1: Open project directory using WSL, configure environment, upgrade pip, and install dependencies from requirements.txt

1. Press Ctrl + Shift + P in the IDE (to open the command palette).
2. Search for and select: WSL: Connect to WSL
3. Click Open Folder, and select your project path inside WSL (e.g., /mnt/c/Final_Project)
4. Verify connection status: check the lower-left corner status bar to see if it shows green or says WSL: Ubuntu
5. Before installing MuJoCo or Unitree software, install the Linux packages required for Python virtual environments, C/C++ compilation, CMake/Ninja builds, OpenGL rendering, GLFW window management, and WSLg graphics.
```bash
sudo apt update && sudo apt install -y \
    python3-venv \
    python3-dev \
    build-essential \
    git \
    cmake \
    ninja-build \
    pkg-config \
    libglfw3 \
    libglfw3-dev \
    libgl1-mesa-dev \
    libegl1-mesa-dev \
    libxinerama-dev \
    libxcursor-dev \
    libxrandr-dev
```

6. Create a project-local virtual environment so this workshop does not interfere with other Python installations.

```bash
cd /mnt/c/Final_Project

python3 -m venv .venv
source .venv/bin/activate
```

7. Upgrade pip and install dependencies from requirements.txt:

```bash
python -m pip install --upgrade pip setuptools wheel
pip install -r requirements.txt
```

8. Register the current environment as a Jupyter kernel (so the IDE can see it):
```bash
python3 -m ipykernel install --user --name=my_kernel_name --display-name "Python (WSL Env)"
```

9. Press Ctrl + Shift + P in the IDE, type and select: Developer: Reload Window

#### Step 2: Clone the official third-party Unitree MuJoCo repository (if external/unitree_mujoco does not exist)

In [ ]:
import os
external_dir = os.path.join("external", "unitree_mujoco")
if not os.path.exists(external_dir):
    print("Cloning unitree_mujoco repository...")
    os.makedirs("external", exist_ok=True)
    !git clone https://github.com/unitreerobotics/unitree_mujoco.git {external_dir}
else:
    print(f"'{external_dir}' already exists, skipping clone.")

#### Step 3: Verify successful installation of major libraries

In [ ]:
import sys
print(f"Python Version: {sys.version}")
try:
    import mujoco
    print(f"MuJoCo Version: {mujoco.__version__}")
except ImportError:
    print("Error: MuJoCo is not installed.")
try:
    import torch
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")
except ImportError:
    print("Error: PyTorch is not installed.")
try:
    import gymnasium as gym
    print(f"Gymnasium Version: {gym.__version__}")
except ImportError:
    print("Error: Gymnasium is not installed.")

### 2. Model Joint Audit & Gesture Calibration [Phase 0]
Before starting reinforcement learning, we must first generate the 29-DOF fixed-base G1 model, audit its wrist and finger Actuators, and manually calibrate the target joint angle vectors for the three gestures.
- **Executable Files**: [create_fixed_base_g1.py](./src/create_fixed_base_g1.py) and [g1_model_audit.py](./src/g1_rl/g1_model_audit.py)
- **Tasks**: Load the G1 robot hand XML model, print all Joint and Actuator names with their limits, and manually calibrate & save the target joint angle vectors for the following three gestures (structured as `[wrist_roll, wrist_pitch, wrist_yaw, thumb, finger_1, finger_2]`):
  1. **Thumbs Up**: Thumb extended, main fingers flexed, wrist oriented up.
     - **Target Vector**: `[-1.57, 0.0, 0.0, 1.0, 0.0, 0.0]`
     - **Motion Description**: Wrist roll `L_WRIST_ROLL` is `-1.57 rad` ($-90^\circ$) to point the thumb upwards; the virtual thumb is fully extended (`1.0`); index and middle fingers are tightly flexed (`0.0`).
  2. **Open/Stop**: All digits extended, palm facing forward.
     - **Target Vector**: `[1.57, 0.0, 1.6144, 0.0, 1.0, 1.0]`
     - **Motion Description**: Wrist roll `L_WRIST_ROLL` is `+1.57 rad` ($+90^\circ$) and wrist yaw `L_WRIST_YAW` is `+1.6144 rad` ($+92.5^\circ$) to make the palm face fully forward; the virtual thumb is at position `0.0`; index and middle fingers are fully extended (`1.0`).
  3. **Thumbs Down**: Thumb extended, main fingers flexed, wrist oriented down.
     - **Target Vector**: `[1.57, 0.0, 0.0, 1.0, 0.0, 0.0]`
     - **Motion Description**: Wrist roll `L_WRIST_ROLL` is `+1.57 rad` ($+90^\circ$) to point the thumb downwards; the virtual thumb is fully extended (`1.0`); index and middle fingers are tightly flexed (`0.0`).

> **3-Digit Hand Rule**: Unitree G1 uses a 3-digit hand structure comprising **two main fingers + one thumb**. Using a 5-digit human hand model or human-hand rendering is strictly prohibited.

| Phase | Step Name | Arm & Wrist State | 3-Digit Finger Poses | Success Criterion |
| :--- | :--- | :--- | :--- | :--- |
| **0. Initial State** | Keep Pose | Left arm extended horizontally, wrist in neutral<br>(`L_WRIST_ROLL` = `0.0 rad`, back of hand outward) | 3 digits fully flexed (fist)<br>(virtual values `[0.0, 0.0, 0.0]`) | Joint positions stabilized and velocities below threshold |
| **1. Raise Thumb** | Thumbs Up | Maintain horizontal, wrist roll $-90^\circ$<br>(`L_WRIST_ROLL` = `-1.57 rad`, thumb pointing up) | Thumb extended (`1.0`), main fingers flexed (`0.0`)<br>(virtual values `[1.0, 0.0, 0.0]`) | Wrist & thumb joint angles match targets and pose is stable |
| **2. Open Palm** | Open/Stop | Maintain horizontal, wrist roll $+90^\circ$ and yaw $+92.5^\circ$<br>(`L_WRIST_ROLL` = `+1.57 rad`, `L_WRIST_YAW` = `+1.6144 rad`, palm forward) | Thumb extended (`0.0`), main fingers fully extended (`1.0`)<br>(virtual values `[0.0, 1.0, 1.0]`) | 3 digits and wrist angles all reach opening thresholds |
| **3. Return to Fist** | Refist | Maintain horizontal, wrist returns to neutral<br>(`L_WRIST_ROLL` = `0.0 rad`, back of hand outward) | 3 digits return to flexed (fist)<br>(virtual values `[0.0, 0.0, 0.0]`) | 3 digit joint angles return to initial fist states |
| **4. Invert to Dislike** | Thumbs Down | Wrist roll $+90^\circ$<br>(`L_WRIST_ROLL` = `+1.57 rad`, thumb pointing down)<br>*Note: rotated $180^\circ$ from Thumbs Up* | Thumb extended (`1.0`), main fingers flexed (`0.0`)<br>(virtual values `[1.0, 0.0, 0.0]`) | Wrist roll angle reaches target (`+1.57 rad`) and thumb is stably down |
| **5. Reset/Loop** | Reset | Wrist roll returns to neutral<br>(`L_WRIST_ROLL` = `0.0 rad`, back of hand outward) | 3 digits fully open (back to Open/Stop)<br>(virtual values `[0.0, 1.0, 1.0]`) | Wrist and fingers return to Phase 2 (Open/Stop) states |


In [ ]:
# Generate fixed-base G1 model and execute model audit
!python src/create_fixed_base_g1.py

!python src/g1_rl/g1_model_audit.py --xml-path assets/g1_fixed_base/scene_29dof_fixed_base.xml --no-viewer

### 3. Gymnasium Environment Development & Random Action Test [Phase 1]
- **Executable File**: [g1_hand_env.py](./src/g1_rl/g1_hand_env.py)
- **Tasks**:
  * Define continuous state space $s_t$: $s_t = [q_t, \dot{q}_t, q_{\text{target}}(g), q_{\text{target}}(g)-q_t, \text{one\_hot}(g), a_{t-1}]$
  * Define continuous action space $a_t$: joint angle increment control (for wrist and fingers).
  * Implement composite reward function:
    $$r_t = w_p(e_{t-1} - e_t) - w_h E_{\text{hand}} - w_o E_{\text{orientation}} - w_v \|\dot{q}_t\|^2 - w_a \|a_t\|^2 - w_s \|a_t-a_{t-1}\|_2^2 + b_{\text{hold}} I_{\text{hold}} - c_{\text{time}}$$

    Where the weighted pose error $e_t$ is defined as:

    $$e_t = w_{\text{wrist}} \Vert{}q_t^{\text{wrist}} - q_{\text{target}}^{\text{wrist}}\Vert{}_2 + w_{\text{finger}} \Vert{}q_t^{\text{finger}} - q_{\text{target}}^{\text{finger}}\Vert{}_2$$

  * Implement gesture hold detection (e.g., success is triggered when both pose and orientation errors remain below target thresholds for at least 15 consecutive steps).

Run the random action test cell below to verify if the Gymnasium wrapper in [g1_hand_env.py](./src/g1_rl/g1_hand_env.py) functions properly:

In [ ]:
# Test whether the Gymnasium environment initializes and steps properly (random action smoke test)
import os
import sys
sys.path.append(os.path.abspath("src"))
from g1_rl.g1_hand_env import G1HandEnv

try:
    # Instantiate environment
    env = G1HandEnv(xml_path="assets/g1_fixed_base/scene_29dof_fixed_base.xml")
    obs, info = env.reset()
    print("SUCCESS: Environment initialized successfully!")
    print(f"Observation Space Shape: {obs.shape}")
    print(f"Action Space Shape: {env.action_space.shape}")
    print(f"Initial target gesture: {info.get('target_gesture')}")
    
    # Execute 1 random step
    action = env.action_space.sample()
    next_obs, reward, terminated, truncated, step_info = env.step(action)
    print("SUCCESS: Environment stepped successfully!")
    print(f"Step Reward: {reward}")
    print(f"Next Obs Shape: {next_obs.shape}")
    print(f"Reward info components: {step_info.get('reward_info')}")
except Exception as e:
    print("Environment test failed. (This is normal if xml model or logic is in placeholder status)")
    print("Error details:", e)

### 4. PPO Algorithm Architecture & Network Design [Phase 2]
- **Executable Files**: [actor_critic_network.py](./src/Thumbs_Robot/actor_critic_network.py), [rollout_buffer.py](./src/Thumbs_Robot/rollout_buffer.py), [agent.py](./src/Thumbs_Robot/agent.py)

* Project Workflow Diagram


```text

======================= Phase 1: Rollout Collection (On-Policy Sampling & Env Interaction) =======================
Current State s_t (includes joint states, target gesture, previous action a_{t-1})
  │
  ├────────────────────────────────────────────────┐
  ▼ (Actor Network)                          ▼ (Critic Network)
Mean μ_θ(s_t) & Std σ_θ(s_t)                 State Value Estimate V_ϕ(s_t)
  │                                               │
  ▼ (Gaussian Sampling)                           │
Continuous Action a_raw ~ N(μ_θ, σ_θ)             │
  │                                               │
  ▼ (Action Safety Boundary Clipping)             │
Safe Action a_t = clip(a_raw, a_min, a_max)       │
  │                                               │
  ▼ (Step Action in MuJoCo Sim Environment)        │
Compute smoothness penalty & hold bonus, get r_t   │
Obtain next state s_{t+1} & terminated/truncated   │
  │                                               │
  ▼                                               ▼
Store (s_t, a_t, r_t, log_prob_old, V_ϕ(s_t), terminated, truncated) into Buffer
                                (Repeat for N rollout steps)

======================= Phase 2: Advantage Estimation & Bootstrap Handling =======================
Read trajectory data from Rollout Buffer (States, Unclipped_Actions, Rewards, Masks...)
  │
  ▼ (Bootstrapping Treatment)
Compute value of the terminal state V_ϕ(s_{N+1}):
  ├── If terminated = True  ──> Mask = 0, no future value accumulation
  └── If truncated = True   ──> Mask = 1, Bootstrap using V_ϕ(s_{N+1})
  │
  ▼ (Generalized Advantage Estimation GAE-λ)
Compute A_t (GAE) and Return Target R_t = A_t + V_ϕ(s_t)

======================= Phase 3: PPO Policy & Value Network Updates =======================
Shuffle data, split into Mini-batches (run updates for K epochs)
  │
  ├────────────────────────────────────────┐
  ▼ (Current Actor Network)                 ▼ (Current Critic Network)
Compute new log_prob & entropy H(π_θ)      Predict new value V_ϕ(s_t)
  │                                        │
  ▼ (Calculate probability ratio ρ_t)      │
ρ_t = exp(log_prob(a_raw) - log_prob_old(a_raw))
  │                                        │
  ▼ (Advantage Mini-batch Normalization)   │
A_norm = (A_batch - mean) / (std + 1e-8)   │
  │                                        │
  ├────────────────────────────────────────┼────────────────────────────────────────┐
  ▼ (Actor Loss Computation)                ▼ (Critic Loss Computation)              ▼ (Entropy Regularization)
L_actor = -min(ρ_t*A_norm,                 L_critic = (R_t - V_ϕ(s_t))²             H(π_θ)
               clip(ρ_t, 1-ε, 1+ε)*A_norm)
  │                                        │                                         │
  └───────────────────────────────────────┬─┴────────────────────────────────────────┘
                                          ▼
                          Compute Weighted Total Loss
                          L_total = L_actor + c₁*L_critic - c₂*H(π_θ)
                                          │
                                          ▼
                              Backpropagation
                                          │
                                          ▼ (Gradient Clipping)
                              L2 Norm Gradient Clipping
                                          │
                                          ▼
                              Update Actor (θ) and Critic (ϕ) weights

======================= Phase 4: Diagnostics & Logging =======================
Calculate and log metrics to Console / CSV / TensorBoard:
- Policy divergence (approx_kl)
- Value function explained variance (explained_variance)
- Policy average standard deviation (mean_actor_std)
- Loss components (actor_loss, critic_loss, entropy)

```

- **The table below shows the mapping between mathematical concepts, algorithmic logic, code variables, and real-time logs**

| Math Concept (MDP Formula) | Algorithm Logic | Code Variable | Console / CSV Log Field |
| :--- | :--- | :--- | :--- |
| **Current State** $s_t$ | Physics and task observation vector | `obs` / `state` | `pose_error`, `orientation_error` |
| **Action Distribution** $\pi_\theta(a \mid s_t)$ | Gaussian mean and standard deviation | `dist.mean` / `dist.stddev` | `actor_mean`, `actor_std` |
| **Sampled Action** $a_{raw}$ | Exploration sample from Gaussian | `raw_action` / `action_unclipped` | `action_sample` |
| **Safe Action** $a_t$ | Clip action to physical joint limits | `clipped_action` | `action_clipped` |
| **Smoothness Penalty** $r_{smooth}$ | Penalize square difference of adjacent actions | `r_smooth` / `r_smooth_delta` | `smoothness_penalty`, `smoothness_delta_penalty` |
| **Total Reward** $r_t$ | Composite multi-objective reward | `reward` / `total_reward` | `reward` / `reward_total` |
| **State Value** $V_\phi(s_t)$ | Critic expected reward prediction | `value` / `value_t` | `value` / `V(s_t)` |
| **Bootstrap Estimate** $V(s_{t+1})$ | Value of next state (handles truncation) | `next_value` / `next_val` | `next_value` |
| **Return Target** $R_t$ (TD Target) | Bellman target estimation (GAE + Value) | `returns` / `td_tgt` | `td_target` |
| **Advantage Estimate** $A_t$ | GAE advantage signal normalized over batch | `advantages` / `adv` | `advantage` |
| **Probability Ratio** $\rho_t(\theta)$ | Importance sampling ratio (new / old) | `ratio` | `clip_fraction` (clipping monitor) |
| **Policy Divergence** $KL(\pi_{\theta_{old}} \| \pi_\theta)$ | Monitor KL divergence during updates | `approx_kl` | `approx_kl`  |
| **Policy Loss** $L_{\text{actor}}$ | PPO Clipped Surrogate Loss | `actor_loss` | `actor_loss` |
| **Value Loss** $L_{\text{critic}}$ | Value network Mean Squared Error (MSE) | `critic_loss` | `critic_loss` |
| **Policy Entropy** $\mathcal{H}(\pi_\theta)$ | Information entropy for exploration | `entropy` | `entropy` |
| **Total Loss** $L_{\text{total}}$ | Weighted sum of actor, critic, and entropy | `loss` / `total_loss` | `total_loss` |
| **Explained Variance** $EV$ | Evaluates Critic prediction accuracy | `explained_var` | `explained_variance` |

- **Component Responsibilities**:
  1. **Actor Network**: Takes $s_t$, predicts the mean and standard deviation of continuous action distributions, and samples using Gaussian distribution. During optimization, it computes the log probabilities and entropy of the new policy to constrain updates and sustain exploration.
  2. **Critic Network**: Takes $s_t$, predicts the expected state value $V(s_t)$. When episodes are truncated by time limits, it predicts the terminal bootstrap value to eliminate boundary bias.
  3. **Rollout Buffer**: Stores on-policy trajectory data (States, Actions, Log_probs, Rewards, Values, Terminated, Truncated), handles terminal type checks to compute GAE advantages and returns, and performs advantage normalization during updates.


### 5. PPO Components Verification [Phase 3]
- **Executable File**: [smoke_test.py](./src/Thumbs_Robot/smoke_test.py)
- **Tasks**:
  Complete PyTorch network forward propagation (outputting Gaussian mean/std and state value $V$), GAE advantage estimation, and PPO Clipped updates.
  Run the smoke test script below to verify neural network dimensions, buffer sampling, and optimization update steps are functioning correctly without NaN/Inf values.

In [ ]:
# Run smoke test to verify PPO components and network architecture correctness
!python src/Thumbs_Robot/smoke_test.py

### 6. Four-in-One Mapping Logs (Math-to-Code-to-Log) [Phase 4]
- **Tasks**: Implement step-level and optimization update CSV logs, print aligned mathematical relations in the console in real-time, and verify that the "Math $\rightarrow$ Algorithm $\rightarrow$ Code $\rightarrow$ Log" alignment is precisely met.
- **Verification**: Run the training smoke test, monitor the update-level table log printed in the main console (including `Loss_A`, `Loss_C`, `Entropy`, and gesture completion summaries), and inspect `episode_log.csv` quietly written to `results/ppo_config_a/` (containing episode-level metrics like `Reward`, `Final Pose Error`, `Hold Duration`, and `Safety Violation`) to confirm the alignment.


In [ ]:
# Core configuration name: you can change this to switch experimental configs (e.g., "ppo_config_b", "ppo_config_c")
CONFIG_NAME = "ppo_config_smoke_test"

# Run smoke update test, verify four-in-one console logs and CSV files
!python src/Thumbs_Robot/train_thumbs.py --smoke-test --results-dir results/{CONFIG_NAME}


In [ ]:
# View details in the quietly written episode_log.csv
import os
import pandas as pd

episode_log_path = f"results/{CONFIG_NAME}/episode_log.csv"
if os.path.exists(episode_log_path):
    df = pd.read_csv(episode_log_path)
    print(f"SUCCESS: Read {len(df)} episode records. The latest 10 rows are:")
    display(df.tail(10))
else:
    print(f"Error: Log file not found at {episode_log_path}. Please complete training first.")


### 7. Target-Conditioned Headless Training [Phase 5]
- **Executable File**: [train_thumbs.py](./src/Thumbs_Robot/train_thumbs.py)
- **Tasks**: Launch the formal training of the Unified Target-Conditioned Policy, training an Actor-Critic (PPO) agent to master Thumbs Up, Open/Stop, and Thumbs Down simultaneously.
- **Notes**: Training logs and model weights will be saved periodically in `results/` and `models/` directories.

In [ ]:
# Launch formal continuous PPO target-conditioned multi-gesture training
# Training logs will be written to the results/ directory

# HYPERPARAMETERS CONFIGURATION

CONFIG_NAME = "ppo_config_a5"

# 1. Start formal training within the Jupyter Kernel
""" import argparse
from Thumbs_Robot.train_thumbs import run_training
# Define your custom parameters for experiment

custom_args = argparse.Namespace(
    seed=666,
    lr=1e-4,                    # Adjust learning rate (e.g., lower from 3e-4 to 1e-4 to stabilize training)
    gamma=0.99,
    gae_lambda=0.95,
    clip_epsilon=0.1,           # Adjust PPO clip epsilon for more conservative policy updates
    ppo_epochs=10,
    batch_size=64,
    rollout_length=2048,
    max_total_steps=100000,     # Formal training steps
    results_dir=f"results/{CONFIG_NAME}", # Keep in sync with CONFIG_NAME
    cpu=False,
    smoke_test=False,
    hidden_dim=256,
    initial_log_std=-0.8        # Adjust initial log std to narrow initial exploration and speed up convergence
)


run_training(custom_args) 
"""

# 2. Launch training via command-line in WSL terminal

!python src/Thumbs_Robot/train_thumbs.py --results-dir results/{CONFIG_NAME} --lr 3e-4 --clip-epsilon 0.2 --initial-log-std -1.0 --entropy-coef 0.001 --max-total-steps 250000



In [ ]:
# View detailed training episode log metrics
import os
import pandas as pd

episode_log_path = f"results/{CONFIG_NAME}/episode_log.csv"
if os.path.exists(episode_log_path):
    df = pd.read_csv(episode_log_path)
    print(f"SUCCESS: Read {len(df)} episode records. The latest 10 rows are:")
    display(df.tail(10))
else:
    print(f"Error: Log file not found at {episode_log_path}. Please execute the training cell first.")


### 8. One-click Evaluation & 3D Render Visualization [Phase 6]
- **Executable Files**: [evaluate_thumbs.py](./src/Thumbs_Robot/evaluate_thumbs.py) and [render_thumbs.py](./src/Thumbs_Robot/render_thumbs.py)
- **Tasks**: Load the best checkpoint weights, perform a deterministic evaluation run, and launch the MuJoCo 3D interactive viewer to visualize the dynamic hand-gesture controller.

In [ ]:
# Run deterministic policy evaluation and calculate success metrics
!python src/Thumbs_Robot/evaluate_thumbs.py --checkpoint models/{CONFIG_NAME}/{CONFIG_NAME}_best.pt --output_dir results/{CONFIG_NAME}_evaluation


### 9. Results Visualization & Presentation Prep [Phase 7]
- **Executable File**: [plot_results.py](./src/Thumbs_Robot/plot_results.py)
- **Tasks**: Plot the training convergence curves for returns, success rate, optimization losses, and policy entropy.
Run the cell below to plot the training results and display them directly in the notebook.

- **3D Interactive Simulation Command**: `python3 src/Thumbs_Robot/render_thumbs.py --checkpoint models/{CONFIG_NAME}/{CONFIG_NAME}_best.pt`

In [ ]:
# Plot convergence curves and output images
!python src/Thumbs_Robot/plot_results.py --results-dir results/{CONFIG_NAME}
